# 00 — Shared configuration

Settings and helper functions used by every notebook in the pipeline (owner: Member 1).

Other notebooks load this file with `%run -i ./00_config.ipynb`, so keep this notebook
**definitions only**: no crawling, no long-running code.

What lives here:
- the assigned arXiv ID ranges and the data / log folders,
- rate-limit settings (arXiv: 3 s between calls, Semantic Scholar: 1 s),
- ID helpers (`split_version`, `to_folder_id`, `paper_dir`),
- JSON and log helpers (`write_json`, `read_json`, `log_paper`, `done_ids`).

In [ ]:
import csv
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

## 1. Run mode

`RUN_TEST` controls the small test cell at the end of notebooks 01–04.
When `05_run_pipeline.ipynb` sets `RUN_TEST = False` before `%run -i`, that value is kept;
when a notebook is opened on its own, it defaults to `True`.

In [ ]:
if "RUN_TEST" not in globals():
    RUN_TEST = True

## 2. Assigned ID ranges

Ranges given by the Lab Instructors, one per member (5,000 IDs each, 20,000 in total).
Both ends are inclusive and share the same `YYMM` prefix. The instructors wrote them as
month + number (`2026-09  32834`), which is the arXiv ID `2609.32834`.

In [ ]:
ID_RANGES = [
    ("2609.32834", "2609.37833"),   # 2026-09  32834 .. 37833
    ("2605.17338", "2605.22337"),   # 2026-05  17338 .. 22337
    ("2605.07338", "2605.12337"),   # 2026-05  07338 .. 12337
    ("2603.20580", "2603.25579"),   # 2026-03  20580 .. 25579
]

## 3. Paths

On Google Colab the data goes to Google Drive, so a disconnect does not lose the crawl.
Locally it goes to `<repo>/data/raw/` (git-ignored).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/arxiv-project")
else:
    # Notebooks live in src/, so the repo root is one level up
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "raw"   # one folder per paper: 2310-12345/
LOG_DIR = PROJECT_ROOT / "logs"
PAPER_LOG = LOG_DIR / "papers.csv"          # one row per paper, used for resume + statistics

DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

## 4. Rate limits and HTTP settings

- arXiv API, OAI-PMH and `/src/` downloads: at least 3 s between calls, one connection.
- Semantic Scholar: 1 request/s (read the key from the `S2_API_KEY` environment variable or Colab Secrets, never hard-code it).

In [ ]:
ARXIV_DELAY = 3.0          # seconds between arXiv requests
ARXIV_BATCH_SIZE = 100     # IDs per arXiv API call
ARXIV_RETRIES = 3          # retries after HTTP 429 before giving up
ARXIV_BACKOFF = 30.0       # first wait after HTTP 429, doubled on each retry
S2_DELAY = 1.0             # seconds between Semantic Scholar requests

USER_AGENT = "HCMUS-IntroDS-arxiv-project/0.1 (student course project)"

S2_API_KEY = os.environ.get("S2_API_KEY")
if IN_COLAB and not S2_API_KEY:
    try:
        from google.colab import userdata
        S2_API_KEY = userdata.get("S2_API_KEY")
    except Exception:
        S2_API_KEY = None

## 5. ID helpers

arXiv uses two ID formats:
- new style (2007+): `2310.12345`, with 4 digits after the dot before 2015 and 5 digits from 2015;
- old style (before 2007): `archive/YYMMNNN`, e.g. `hep-th/9901001`.

Either may carry a version suffix (`v2`). IDs are always stored as originally given;
only the **folder name** is changed, because `.` and `/` are awkward in paths.

In [ ]:
NEW_ID_RE = re.compile(r"^(\d{4})\.(\d{4,5})$")
OLD_ID_RE = re.compile(r"^([a-z\-]+(?:\.[A-Z]{2})?)/(\d{7})$")
VERSION_RE = re.compile(r"^(.+?)v(\d+)$")


def split_version(arxiv_id):
    """'2310.12345v2' -> ('2310.12345', 2); '2310.12345' -> ('2310.12345', None)."""
    m = VERSION_RE.match(arxiv_id)
    if m:
        return m.group(1), int(m.group(2))
    return arxiv_id, None


def to_folder_id(arxiv_id):
    """Folder name for a paper: '2310.12345' -> '2310-12345', 'hep-th/9901001' -> 'hep-th_9901001'."""
    base, _ = split_version(arxiv_id)
    return base.replace(".", "-").replace("/", "_")


def paper_dir(arxiv_id):
    """Folder that holds all files of one paper (created if missing)."""
    path = DATA_DIR / to_folder_id(arxiv_id)
    path.mkdir(parents=True, exist_ok=True)
    return path


def to_iso(dt):
    """datetime -> ISO 8601 string in UTC (None stays None)."""
    if dt is None:
        return None
    return dt.astimezone(timezone.utc).isoformat()

## 6. JSON helpers

`write_json` writes to a temporary file first and then renames it, so an interrupted run
never leaves a half-written file behind.

In [ ]:
def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## 7. Paper log

`logs/papers.csv` has one row per processed paper. The pipeline reads it to skip finished
papers when it is restarted, and Member 4 reads it to compute the report statistics.
Column format agreed by the team:

In [ ]:
LOG_FIELDS = [
    "id",            # arXiv ID without version
    "versions",      # number of versions on arXiv
    "versions_ok",   # versions whose source was downloaded
    "size_before",   # bytes before figure removal
    "size_after",    # bytes after figure removal
    "n_refs",        # references returned by Semantic Scholar
    "n_arxiv_refs",  # references that have an arXiv ID
    "status",        # ok / partial / not_found / error
    "seconds",       # processing time for this paper
    "error",         # error message, empty if none
    "logged_at",     # UTC time the row was written
]


def log_paper(row):
    """Append one paper's result to logs/papers.csv (missing fields are left empty)."""
    new_file = not PAPER_LOG.exists()
    row = {**row, "logged_at": to_iso(datetime.now(timezone.utc))}
    with open(PAPER_LOG, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS, extrasaction="ignore")
        if new_file:
            writer.writeheader()
        writer.writerow(row)


def done_ids(statuses=("ok", "not_found")):
    """IDs already finished in an earlier run, so a restart can skip them."""
    if not PAPER_LOG.exists():
        return set()
    with open(PAPER_LOG, newline="", encoding="utf-8") as f:
        return {r["id"] for r in csv.DictReader(f) if r["status"] in statuses}

In [ ]:
CONFIG_LOADED = True
print(f"Config loaded | Colab: {IN_COLAB} | data: {DATA_DIR} | ranges: {len(ID_RANGES)}")